# Imports

In [ ]:
import math
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Setup

LLM used to setup access and dataset

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

print(f'Seed   : {SEED}')
print(f'Device : {DEVICE}')

Seed   : 42
Device : cuda


In [ ]:
## Kaggle access
from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
%pip install kaggle -q
print('Kaggle configured')

Saving kaggle.json to kaggle.json
Kaggle configured


In [ ]:
## Download dataset
!kaggle competitions download -c models-from-scratch -p ./data/
!unzip -q ./data/models-from-scratch.zip -d ./data/
print('Dataset downloaded')
!ls ./data/

100% 5.12G/5.12G [00:28<00:00, 195MB/s]

Dataset downloaded
Food10	  models-from-scratch.zip  test.csv
food-101  sample_submission.csv    train.csv.csv


# Task 1

## Convolution neural network

In [ ]:
input_csv = "/content/data/train.csv.csv"
output_csv = "/content/data/train_fixed.csv"


df = pd.read_csv(input_csv)
first_col = df.columns[0]

def fix_path(p):
    p = str(p).replace("/", "\\").strip()

    parts = [x for x in p.split("\\") if x]   # remove empty parts

    # keep only: class_name/file_name
    if len(parts) < 2:
        raise ValueError(f"Invalid path: {p}")

    class_name = parts[-2]
    file_name = parts[-1]

    return f"Food10\\{class_name}\\{file_name}"

df[first_col] = df[first_col].apply(fix_path)
df.to_csv(output_csv, index=False)

print(df.head(10).to_string(index=False))
print(f"\nSaved to {output_csv}")

                          image_path        label_name
Food10\hot_and_sour_soup\1002782.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1003688.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1008185.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1008701.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1008924.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1008925.jpg hot_and_sour_soup
 Food10\hot_and_sour_soup\100938.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1010638.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1016365.jpg hot_and_sour_soup
Food10\hot_and_sour_soup\1019649.jpg hot_and_sour_soup

Saved to /content/data/train_fixed.csv


In [ ]:
class Food10Dataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df = df
        self.base_dir = base_dir
        self.transform = transform
        self.path_col = df.columns[0]
        self.label_col = df.columns[1]
        self.classes = sorted(df[self.label_col].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.base_dir, str(row[self.path_col]).replace("\\", "/"))
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.class_to_idx[row[self.label_col]]
        return image, label


csv_path = "/content/data/train_fixed.csv"
base_dir = "/content/data/Food10"
df = pd.read_csv(csv_path)

# Transformation for train
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.2)
])

#Transformations for val
transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])

])

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

True
Tesla T4
12.8


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


df_train, df_val = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[df.columns[1]]
)

# DataLoaders
train_dataset = Food10Dataset(df_train, base_dir, transform=transform_train)
val_dataset   = Food10Dataset(df_val, base_dir, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = models.resnet18(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, 10)
)
model = model.to(device)   # 10 classes pour Food10

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
#optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    correct_train = 0
    total_train = 0
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad() #tout les batchs ont un gradient différent
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total_train += labels.size(0) #taille des batch
        correct_train += (predicted == labels).sum().item()

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0) #taille des batchs
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    train_acc = 100 * correct_train / total_train

    print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.4f}% | Val Acc: {val_acc:.2f}%")

    best_val_acc = 0.0
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


KeyboardInterrupt: 

In [ ]:
class Food10TestDataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df = df
        self.base_dir = base_dir
        self.transform = transform
        self.path_col = df.columns[1]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row[self.path_col])
        if filename.startswith("data\\food-101\\images\\data\\"):
            filename = filename.replace("data\\food-101\\images\\data\\", "")
        path = os.path.join(self.base_dir, filename.replace("\\", "/"))
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

In [ ]:
model = models.resnet18(pretrained=False)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, 10)
)
model.load_state_dict(torch.load("best_model.pth"))
model.to(device)
model.eval()

base_dir = "/content/data/food-101/"

test_df = pd.read_csv("/content/data/test.csv")
test_dataset = Food10TestDataset(test_df, base_dir, transform=transform_val)

test_loader = DataLoader(test_dataset, batch_size=70, shuffle=False)

predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.cpu().numpy())

idx_to_class = {v: k for k, v in train_dataset.class_to_idx.items()}
predicted_labels = [idx_to_class[p] for p in predictions]

#Create the submission DataFrame
submission = pd.DataFrame({
    "original_index": test_df["original_index"],  # même ordre que le sample
    "label_name": predicted_labels
})



FileNotFoundError: [Errno 2] No such file or directory: 'best_model.pth'

In [ ]:
# Save CSV
submission.to_csv("/content/data/submission.csv", index=False)

## Vision transformer

### Constants and data preprocessing

In [ ]:
IMG_SIZE    = 224
BATCH_SIZE  = 32
VAL_SPLIT   = 0.2
NUM_CLASSES = 10
BASE_DIR    = './data/Food10'

# Fixeing the path
input_csv  = './data/train.csv.csv'
output_csv = './data/train_fixed.csv'
df = pd.read_csv(input_csv)
first_col = df.columns[0]
def fix_path(p):
    p = str(p).replace('/', '\\').strip()
    parts = [x for x in p.split('\\') if x]
    if len(parts) < 2:
        raise ValueError(f'Invalid path: {p}')
    return f'Food10\\{parts[-2]}\\{parts[-1]}'
df[first_col] = df[first_col].apply(fix_path)
df.to_csv(output_csv, index=False)
print(f'CSV fixé : {len(df)} lignes')

# Loading images from the csv
class Food10Dataset(Dataset):
    def __init__(self, df, base_dir, transform=None):
        self.df           = df.reset_index(drop=True)
        self.base_dir     = base_dir
        self.transform    = transform
        self.path_col     = df.columns[0]
        self.label_col    = df.columns[1]
        self.classes      = sorted(df[self.label_col].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.base_dir, str(row[self.path_col]).replace('\\', '/'))
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        label = self.class_to_idx[row[self.label_col]]
        return image, label

# Splittin training and validation set
df_full = pd.read_csv(output_csv)
df_train, df_val = train_test_split(
    df_full,
    test_size    = VAL_SPLIT,
    random_state = SEED,
    stratify     = df_full[df_full.columns[1]]
)
print(f'Train : {len(df_train)} images')
print(f'Val   : {len(df_val)} images')

# Compute mean and std on training set
temp_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

temp_dataset = Food10Dataset(df_train, BASE_DIR, transform=temp_transforms)
temp_loader  = DataLoader(temp_dataset, batch_size=64, shuffle=False, num_workers=2)

mean = torch.zeros(3)
std  = torch.zeros(3)
for imgs, _ in temp_loader:
    for c in range(3):
        mean[c] += imgs[:, c].mean()
        std[c]  += imgs[:, c].std()
mean /= len(temp_loader)
std  /= len(temp_loader)

MEAN = mean.tolist()
STD  = std.tolist()
print(f'mean = [{MEAN[0]:.3f}, {MEAN[1]:.3f}, {MEAN[2]:.3f}]')
print(f'std  = [{STD[0]:.3f}, {STD[1]:.3f}, {STD[2]:.3f}]')

# Transform training set
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
    transforms.RandomErasing(p=0.2)
])

# Transform validation set (only normalization)
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

# Create training and validation datloaders
train_dataset = Food10Dataset(df_train, BASE_DIR, transform=train_transforms)
val_dataset   = Food10Dataset(df_val,   BASE_DIR, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Maps class name to indices
CLASS_NAMES   = train_dataset.classes
class_to_idx  = train_dataset.class_to_idx
idx_to_class  = {v: k for k, v in class_to_idx.items()}

print(f'Train : {len(train_dataset)} images — {len(train_loader)} batches')
print(f'Val   : {len(val_dataset)} images — {len(val_loader)} batches')
print(f'Classes : {CLASS_NAMES}')

CSV fixé : 7500 lignes
Train : 6000 images
Val   : 1500 images
mean = [0.533, 0.433, 0.334]
std  = [0.272, 0.276, 0.275]
Train : 6000 images — 188 batches
Val   : 1500 images — 47 batches
Classes : ['baby_back_ribs', 'beef_tartare', 'cheesecake', 'crab_cakes', 'dumplings', 'french_onion_soup', 'guacamole', 'hot_and_sour_soup', 'pork_chop', 'sushi']


### Patch embedding

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels=3, embedding_dim=256):
        super().__init__()
        assert img_size % patch_size == 0, 'img_size must be divisible by patch_size'
        self.num_patches = (img_size // patch_size) ** 2
        self.projection  = nn.Conv2d(in_channels, embedding_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x

### Positional encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, num_patches, embedding_dim, dropout=0.1):
        super().__init__()
        self.cls_token     = nn.Parameter(torch.zeros(1, 1, embedding_dim))
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, embedding_dim) * 0.02)
        self.dropout       = nn.Dropout(dropout)

    def forward(self, x):
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = x + self.pos_embedding
        return self.dropout(x)

### Multi-Head Self-Attention

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(embed_dim, embed_dim * 3)
        self.proj      = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))

### Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, mlp_ratio=4, dropout=0.1):
        super().__init__()
        hidden_dim = embed_dim * mlp_ratio
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

### Encoder Block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn  = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn   = FeedForward(embed_dim, mlp_ratio, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

### ViT model

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size, patch_size, embedding_dim, depth, num_heads,
                 mlp_ratio=4, dropout=0.1, num_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels=3, embedding_dim=embedding_dim)
        num_patches      = self.patch_embed.num_patches
        self.pos_enc     = PositionalEncoding(num_patches, embedding_dim, dropout)
        self.blocks      = nn.ModuleList([
            TransformerEncoderBlock(embedding_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Conv2d):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.patch_embed(x)
        x = self.pos_enc(x)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.head(x[:, 0])

### Training method

In [ ]:
def train_one_config(config, train_loader, val_loader):
    model = VisionTransformer(
        img_size      = IMG_SIZE,
        patch_size    = config['patch_size'],
        embedding_dim = config['embed_dim'],
        depth         = config['depth'],
        num_heads     = config['num_heads'],
        dropout       = 0.1,
        num_classes   = NUM_CLASSES
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.1)

    WARMUP = 5
    def lr_lambda(epoch):
        if epoch < WARMUP:
            return (epoch + 1) / WARMUP
        progress = (epoch - WARMUP) / (EPOCHS - WARMUP)
        return 0.5 * (1 + math.cos(math.pi * progress))

    scheduler    = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    best_val_acc = 0.0
    history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)
            train_correct += (outputs.argmax(1) == labels).sum().item()
            train_total += imgs.size(0)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                val_correct += (outputs.argmax(1) == labels).sum().item()
                val_total += imgs.size(0)

        t_loss = train_loss / train_total
        t_acc  = train_correct / train_total
        v_loss = val_loss / val_total
        v_acc  = val_correct / val_total

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), f'best_vit_{config["embed_dim"]}_{config["depth"]}.pth')

        scheduler.step()
        print(f'Epoch {epoch:2d}/{EPOCHS} | Train loss: {t_loss:.4f} acc: {t_acc:.4f} | Val loss: {v_loss:.4f} acc: {v_acc:.4f} | LR: {scheduler.get_last_lr()[0]:.2e}')

    return best_val_acc, history, model

### Baysiane optimization

In [ ]:
!pip install optuna -q
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print('Optuna downloaded')

Optuna downloaded


In [ ]:
EPOCHS = 20
import gc

def objective(trial):
    torch.cuda.empty_cache()
    gc.collect()
    config = {
        'patch_size' : trial.suggest_categorical('patch_size', [8, 16]),
        'embed_dim'  : trial.suggest_categorical('embed_dim', [128, 256, 384]),
        'depth'      : trial.suggest_int('depth', 4, 10),
        'num_heads'  : trial.suggest_categorical('num_heads', [4, 8]),
        'lr'         : trial.suggest_float('lr', 1e-4, 5e-4, log=True),
    }
    best_acc, history, _ = train_one_config(config, train_loader, val_loader)
    best_epoch_idx = history['val_acc'].index(best_acc)
    best_train_acc = history['train_acc'][best_epoch_idx]
    trial.set_user_attr("train_acc", best_train_acc)

    return best_acc

In [ ]:
import os
import gc

# Delete if there is already a study
if os.path.exists('optuna_study.db'):
    os.remove('optuna_study.db')
    print('Last study deleted')

torch.cuda.empty_cache()
gc.collect()

study = optuna.create_study(
    direction      = 'maximize',
    sampler        = optuna.samplers.TPESampler(seed=SEED),
    storage        = 'sqlite:///optuna_study.db',
    study_name     = 'vit_search',
    load_if_exists = False
)
study.optimize(objective, n_trials=10)

print(f'\nBest val acc  : {study.best_value:.4f} ({study.best_value*100:.2f}%)')
print(f'Best hyperparameters  : {study.best_params}')

Ancien study supprimé ✅
Epoch  1/15 | Train loss: 2.2494 acc: 0.1698 | Val loss: 2.2085 acc: 0.2107 | LR: 1.05e-04
Epoch  2/15 | Train loss: 2.2047 acc: 0.2030 | Val loss: 2.1215 acc: 0.2627 | LR: 1.58e-04
Epoch  3/15 | Train loss: 2.1464 acc: 0.2473 | Val loss: 2.0884 acc: 0.2747 | LR: 2.10e-04
Epoch  4/15 | Train loss: 2.0812 acc: 0.2692 | Val loss: 1.9776 acc: 0.3373 | LR: 2.63e-04


### Best model training

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

EPOCHS = 40

print(f'Final confuguration : {study.best_params}')
best_acc_final, history_final, best_model = train_one_config(
    study.best_params, train_loader, val_loader
)
print(f'\n→ Best val acc final : {best_acc_final:.4f} ({best_acc_final*100:.2f}%)')

torch.save(best_model.state_dict(), 'best_ViT.pth')
print('Model download as best_ViT.pth')

Config finale : {'patch_size': 8, 'embed_dim': 384, 'depth': 9, 'num_heads': 8, 'lr': 0.0002138277510675073}


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.18 GiB. GPU 0 has a total capacity of 14.56 GiB of which 231.81 MiB is free. Including non-PyTorch memory, this process has 14.33 GiB memory in use. Of the allocated memory 13.99 GiB is allocated by PyTorch, and 215.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

### Submission pipeline

In [ ]:
# Load test set and fixe paths
test_csv = pd.read_csv('./data/test.csv')

def fix_path(p):
    p = p.replace('\\', '/')
    relative = p.split('food-101/images/')[-1]
    return f'./data/Food10/Food10/{relative}'

test_csv['fixed_path'] = test_csv['image_path'].apply(fix_path)
print(f'Founded files : {test_csv["fixed_path"].apply(os.path.exists).sum()} / {len(test_csv)}')

In [ ]:
# Run the best model on the test set
infer_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

all_preds = []
best_model.eval()
with torch.no_grad():
    for path in test_csv['fixed_path']:
        img  = infer_transforms(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)
        pred = best_model(img).argmax(dim=1).item()
        all_preds.append(CLASS_NAMES[pred])

submission = pd.DataFrame({
    'original_index' : test_csv['original_index'].values,
    'label_name'     : all_preds
})
submission.to_csv('submission.csv', index=False)
print(f'submission.csv generated ({len(submission)} lines)')
print(submission['label_name'].value_counts())

In [ ]:
y_true, y_pred = [], []
best_model.eval()
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        outputs = best_model(imgs)
        y_pred.extend(outputs.argmax(1).cpu().numpy())
        y_true.extend(labels.numpy())

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix (Validation Set)')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_final['train_loss'], label='Train loss')
ax1.plot(history_final['val_loss'],   label='Val loss')
ax1.set_title('Loss over epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history_final['train_acc'], label='Train acc')
ax2.plot(history_final['val_acc'],   label='Val acc')
ax2.set_title('Accuracy over epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png')
plt.show()

In [ ]:
## Soumettre sur Kaggle
!kaggle competitions submit -c models-from-scratch -f submission.csv -m "ViT BO optimized"

## Comparaison CNN / Vision Transformer

In [ ]:
  #on load les deux modèles en mode eval :

model_CNN = models.resnet18(pretrained=False) # architecture identique
num_ftrs = model_CNN.fc.in_features
model_CNN.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_ftrs, 10)
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_CNN.to(DEVICE)
model_CNN.load_state_dict(torch.load("/content/data/best_model.pth", map_location=DEVICE))

BEST_CONFIG = {'patch_size': 8, 'embed_dim': 384, 'depth': 9, 'num_heads': 8, 'lr': 0.0002138277510675073}


model_V=VisionTransformer(
        img_size      = IMG_SIZE,
        patch_size    = 8,
        embedding_dim = 384,
        depth         = 9,
        num_heads     = 8,
        dropout       = 0.1,
        num_classes   = NUM_CLASSES
    ).to(DEVICE)

model_V.load_state_dict(torch.load("/content/data/best_vit_384_9.pth", map_location=DEVICE))


model_V.eval()
model_CNN.eval()

y_true = []
pred_vit = []
pred_cnn = []

with torch.no_grad():
    for images, labels in val_loader: #j'utilise le dataset du CNN

        images = images.to(device)
        labels = labels.to(device)

        out_vit = model_V(images)
        out_cnn = model_CNN(images)

        vit_preds = torch.argmax(out_vit, dim=1)
        cnn_preds = torch.argmax(out_cnn, dim=1)

        y_true.extend(labels.cpu().numpy())
        pred_vit.extend(vit_preds.cpu().numpy())
        pred_cnn.extend(cnn_preds.cpu().numpy())

In [ ]:
import numpy as np

# faire la table

y_true = np.array(y_true)
pred_vit = np.array(pred_vit)
pred_cnn = np.array(pred_cnn)

cnn_correct = pred_cnn == y_true
vit_correct = pred_vit == y_true

both_correct = np.sum(cnn_correct & vit_correct)
both_wrong = np.sum(~cnn_correct & ~vit_correct)
cnn_only = np.sum(cnn_correct & ~vit_correct)
vit_only = np.sum(~cnn_correct & vit_correct)

print("Contingency table")
print("-----------------")
print("Both correct:", both_correct)
print("CNN correct only:", cnn_only)
print("ViT correct only:", vit_only)
print("Both wrong:", both_wrong)

Contingency table
-----------------
Both correct: 810
CNN correct only: 261
ViT correct only: 146
Both wrong: 283


# Task 2

In [ ]:
# Task 2 — setup and download data

!pip install timm -q

import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('apikey')

!kaggle competitions download -c models-from-scratch -p ./data/
!unzip -q ./data/models-from-scratch.zip -d ./data/
print('Task 1 (images) downloaded')

!kaggle competitions download -c pre-trained-models -p ./data/task2/
!unzip -q ./data/task2/pre-trained-models.zip -d ./data/task2/
print('Task 2 (test.csv) downloaded')

!ls ./data/
!ls ./data/task2/

100% 5.12G/5.12G [02:06<00:00, 43.5MB/s]

Task 1 (images) downloaded
100% 52.8k/52.8k [00:00<00:00, 2.92MB/s]

Task 2 (test.csv) downloaded
Food10	  models-from-scratch.zip  task2     train.csv.csv
food-101  sample_submission.csv    test.csv
pre-trained-models.zip	sample_submission.csv  test.csv  train.csv.csv


In [ ]:
# Task 2 — data loading

import torch.optim as optim
import timm
import json
from itertools import product

# Constants
TASK2_SAVE_DIR = './task2_outputs'
os.makedirs(TASK2_SAVE_DIR, exist_ok=True)

TASK2_BATCH_SIZE = 64
SEARCH_EPOCHS = 5
FINAL_EPOCHS = 20
NUM_CLASSES = 10
IMG_SIZE = 224

# Set path to Food10 images
TASK2_TRAIN_DIR = './data/Food10/Food10'

print(f'Device: {DEVICE}')
print(f'Task 2 outputs will be saved in: {TASK2_SAVE_DIR}')

# ImageNet normalization (standard for pretrained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Use RandAugment for stronger augmentation in Task 2
task2_train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

task2_val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Create train/val split
raw_dataset_task2 = datasets.ImageFolder(root=TASK2_TRAIN_DIR)
n_total = len(raw_dataset_task2)
n_val = int(n_total * 0.15)
n_train = n_total - n_val

indices = torch.randperm(n_total, generator=torch.Generator().manual_seed(42))
train_idx_task2 = indices[:n_train].tolist()
val_idx_task2 = indices[n_train:].tolist()

print(f'Train: {n_train} | Val: {n_val}')

task2_train_dataset = Subset(
    datasets.ImageFolder(root=TASK2_TRAIN_DIR, transform=task2_train_transforms),
    train_idx_task2
)

task2_val_dataset = Subset(
    datasets.ImageFolder(root=TASK2_TRAIN_DIR, transform=task2_val_transforms),
    val_idx_task2
)

task2_train_loader = DataLoader(
    task2_train_dataset,
    batch_size=TASK2_BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

task2_val_loader = DataLoader(
    task2_val_dataset,
    batch_size=TASK2_BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# Load Task 2 test.csv from pre-trained-models competition (has original_index)
test_csv_task2 = pd.read_csv('./data/task2/test.csv')
print(f'Task 2 test.csv columns: {list(test_csv_task2.columns)}')
print(f'Task 2 test.csv rows: {len(test_csv_task2)}')

# Fix image paths to match this notebook's folder structure
def fix_task2_path(p):
    p = str(p).replace('\\', '/')
    parts = p.split('/')
    class_name = parts[-2]
    filename = parts[-1]
    return f'{TASK2_TRAIN_DIR}/{class_name}/{filename}'

test_csv_task2['fixed_path'] = test_csv_task2['image_path'].apply(fix_task2_path)
found_count = test_csv_task2['fixed_path'].apply(os.path.exists).sum()
print(f'Test files found: {found_count} / {len(test_csv_task2)}')

class Task2TestDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['fixed_path']).convert('RGB')
        return self.transform(img)

task2_test_dataset = Task2TestDataset(test_csv_task2, task2_val_transforms)

task2_test_loader = DataLoader(
    task2_test_dataset,
    batch_size=TASK2_BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# True labels for ensemble validation accuracy calculation
base_dataset_for_targets = datasets.ImageFolder(root=TASK2_TRAIN_DIR)
val_labels_task2 = torch.tensor([base_dataset_for_targets.targets[i] for i in val_idx_task2])

# Class mappings
idx_to_class = {v: k for k, v in base_dataset_for_targets.class_to_idx.items()}

print(f'\nTask 2 Train: {len(task2_train_dataset)} | Val: {len(task2_val_dataset)} | Test: {len(task2_test_dataset)}')

Device: cuda
Task 2 outputs will be saved in: ./task2_outputs
Train: 8500 | Val: 1500
Task 2 test.csv columns: ['original_index', 'image_path']
Task 2 test.csv rows: 2500
Test files found: 2500 / 2500

Task 2 Train: 8500 | Val: 1500 | Test: 2500


In [ ]:
# Task 2 — Mixup/CutMix

def mixup_cutmix(images, labels, num_classes=NUM_CLASSES, alpha=0.2):
    mixed_images = images.clone()
    use_cutmix = torch.rand(1).item() > 0.5
    lam = np.random.beta(alpha, alpha)
    batch_size = mixed_images.size(0)
    index = torch.randperm(batch_size).to(mixed_images.device)

    targets_one_hot = torch.zeros(batch_size, num_classes).to(mixed_images.device)
    targets_one_hot.scatter_(1, labels.unsqueeze(1), 1.0)

    if use_cutmix:
        _, _, h, w = mixed_images.shape
        cut_h, cut_w = int(h * np.sqrt(1-lam)), int(w * np.sqrt(1-lam))
        cx, cy = np.random.randint(w), np.random.randint(h)
        x1, x2 = np.clip(cx-cut_w//2, 0, w), np.clip(cx+cut_w//2, 0, w)
        y1, y2 = np.clip(cy-cut_h//2, 0, h), np.clip(cy+cut_h//2, 0, h)
        mixed_images[:, :, y1:y2, x1:x2] = mixed_images[index, :, y1:y2, x1:x2]
        lam = 1 - ((x2-x1)*(y2-y1)/(h*w))
    else:
        mixed_images = lam * mixed_images + (1-lam) * mixed_images[index]

    return mixed_images, lam * targets_one_hot + (1-lam) * targets_one_hot[index]

print("Mixup/CutMix defined")

Mixup/CutMix defined


In [ ]:
# Task 2 — grid search on EfficientNet-B0

learning_rates = [1e-2, 1e-3, 1e-4]
weight_decays = [1e-3, 1e-4, 1e-5]
grid_results = []

search_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for lr, wd in product(learning_rates, weight_decays):
    print(f"\n{'='*60}")
    print(f"Grid Search | LR={lr} | WD={wd}")
    print(f"{'='*60}")

    search_model = timm.create_model(
        'efficientnet_b0',
        pretrained=True,
        num_classes=NUM_CLASSES
    ).to(DEVICE)

    search_optimizer = optim.AdamW(search_model.parameters(), lr=lr, weight_decay=wd)
    search_scheduler = optim.lr_scheduler.CosineAnnealingLR(search_optimizer, T_max=SEARCH_EPOCHS)

    best_val_acc_search = 0.0
    final_train_acc = 0.0

    for epoch in range(SEARCH_EPOCHS):
        search_model.train()
        correct = 0
        total = 0

        for images, labels in task2_train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = search_model(images)
            loss = search_criterion(outputs, labels)

            search_optimizer.zero_grad()
            loss.backward()
            search_optimizer.step()

            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_acc = correct / total
        final_train_acc = train_acc

        search_model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in task2_val_loader:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                outputs = search_model(images)
                preds = outputs.argmax(dim=1)

                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_acc = correct / total
        best_val_acc_search = max(best_val_acc_search, val_acc)

        search_scheduler.step()
        print(f"  Epoch {epoch+1}/{SEARCH_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    grid_results.append({
        'lr': lr,
        'wd': wd,
        'best_val_acc': best_val_acc_search,
        'final_train_acc': final_train_acc,
        'overfit_gap': final_train_acc - best_val_acc_search
    })

    print(f"  >> Best Val Acc: {best_val_acc_search:.4f}")

    del search_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

grid_df = pd.DataFrame(grid_results).sort_values('best_val_acc', ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("EFFICIENTNET-B0 GRID SEARCH RESULTS")
print("="*70)
print(grid_df.to_string(index=False))

best_lr = float(grid_df.iloc[0]['lr'])
best_wd = float(grid_df.iloc[0]['wd'])

print(f"\nBest EfficientNet hyperparameters: LR={best_lr}, WD={best_wd}")

grid_df.to_csv(os.path.join(TASK2_SAVE_DIR, 'grid_search_effnet_results.csv'), index=False)


Grid Search | LR=0.01 | WD=0.001
  Epoch 1/5 | Train Acc: 0.2354 | Val Acc: 0.3560
  Epoch 2/5 | Train Acc: 0.4807 | Val Acc: 0.5453
  Epoch 3/5 | Train Acc: 0.6362 | Val Acc: 0.6300
  Epoch 4/5 | Train Acc: 0.7162 | Val Acc: 0.7653
  Epoch 5/5 | Train Acc: 0.7840 | Val Acc: 0.8293
  >> Best Val Acc: 0.8293

Grid Search | LR=0.01 | WD=0.0001
  Epoch 1/5 | Train Acc: 0.2005 | Val Acc: 0.3340
  Epoch 2/5 | Train Acc: 0.3993 | Val Acc: 0.4980
  Epoch 3/5 | Train Acc: 0.5469 | Val Acc: 0.5687
  Epoch 4/5 | Train Acc: 0.6486 | Val Acc: 0.6753
  Epoch 5/5 | Train Acc: 0.7173 | Val Acc: 0.7813
  >> Best Val Acc: 0.7813

Grid Search | LR=0.01 | WD=1e-05


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dce09ac9300>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dce09ac9300>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  Epoch 1/5 | Train Acc: 0.2020 | Val Acc: 0.3500


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dce09ac9300>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
   Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7dce09ac9300> 
  Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^
^ ^  ^ ^Exception ignored in: 
 <function _MultiProcessingDataLoaderIter.__del__ at 0x7dce09ac9300>  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
 
     Traceback (most re

  Epoch 2/5 | Train Acc: 0.4626 | Val Acc: 0.5600
  Epoch 3/5 | Train Acc: 0.6094 | Val Acc: 0.6380
  Epoch 4/5 | Train Acc: 0.6966 | Val Acc: 0.7247
  Epoch 5/5 | Train Acc: 0.7546 | Val Acc: 0.8000
  >> Best Val Acc: 0.8000

Grid Search | LR=0.001 | WD=0.001
  Epoch 1/5 | Train Acc: 0.7491 | Val Acc: 0.8780
  Epoch 2/5 | Train Acc: 0.9034 | Val Acc: 0.9060
  Epoch 3/5 | Train Acc: 0.9468 | Val Acc: 0.9187
  Epoch 4/5 | Train Acc: 0.9694 | Val Acc: 0.9307
  Epoch 5/5 | Train Acc: 0.9847 | Val Acc: 0.9327
  >> Best Val Acc: 0.9327

Grid Search | LR=0.001 | WD=0.0001
  Epoch 1/5 | Train Acc: 0.7428 | Val Acc: 0.8687
  Epoch 2/5 | Train Acc: 0.9021 | Val Acc: 0.8973
  Epoch 3/5 | Train Acc: 0.9456 | Val Acc: 0.9280
  Epoch 4/5 | Train Acc: 0.9742 | Val Acc: 0.9347
  Epoch 5/5 | Train Acc: 0.9840 | Val Acc: 0.9333
  >> Best Val Acc: 0.9347

Grid Search | LR=0.001 | WD=1e-05
  Epoch 1/5 | Train Acc: 0.7486 | Val Acc: 0.8827
  Epoch 2/5 | Train Acc: 0.8945 | Val Acc: 0.9060
  Epoch 3/5 | Tr

In [ ]:
# Task 2 — grid search on Swin Transformer

learning_rates_swin = [1e-3, 1e-4]  # Skip 1e-2, already shown to destroy pretrained weights
weight_decays_swin = [1e-3, 1e-4, 1e-5]
grid_results_swin = []

search_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for lr, wd in product(learning_rates_swin, weight_decays_swin):
    print(f"\n{'='*60}")
    print(f"Swin Grid Search | LR={lr} | WD={wd}")
    print(f"{'='*60}")

    search_model = timm.create_model(
        'swin_small_patch4_window7_224',
        pretrained=True,
        num_classes=NUM_CLASSES
    ).to(DEVICE)

    search_optimizer = optim.AdamW(search_model.parameters(), lr=lr, weight_decay=wd)
    search_scheduler = optim.lr_scheduler.CosineAnnealingLR(search_optimizer, T_max=SEARCH_EPOCHS)

    best_val_acc_search = 0.0
    final_train_acc = 0.0

    for epoch in range(SEARCH_EPOCHS):
        search_model.train()
        correct = 0
        total = 0

        for images, labels in task2_train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            outputs = search_model(images)
            loss = search_criterion(outputs, labels)

            search_optimizer.zero_grad()
            loss.backward()
            search_optimizer.step()

            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        train_acc = correct / total
        final_train_acc = train_acc

        search_model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in task2_val_loader:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                outputs = search_model(images)
                preds = outputs.argmax(dim=1)

                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_acc = correct / total
        best_val_acc_search = max(best_val_acc_search, val_acc)

        search_scheduler.step()
        print(f"  Epoch {epoch+1}/{SEARCH_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    grid_results_swin.append({
        'lr': lr,
        'wd': wd,
        'best_val_acc': best_val_acc_search,
        'final_train_acc': final_train_acc,
        'overfit_gap': final_train_acc - best_val_acc_search
    })

    print(f"  >> Best Val Acc: {best_val_acc_search:.4f}")

    del search_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

grid_df_swin = pd.DataFrame(grid_results_swin).sort_values('best_val_acc', ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("SWIN GRID SEARCH RESULTS")
print("="*70)
print(grid_df_swin.to_string(index=False))

best_lr_swin = float(grid_df_swin.iloc[0]['lr'])
best_wd_swin = float(grid_df_swin.iloc[0]['wd'])

print(f"\nBest Swin hyperparameters: LR={best_lr_swin}, WD={best_wd_swin}")

grid_df_swin.to_csv(os.path.join(TASK2_SAVE_DIR, 'grid_search_swin_results.csv'), index=False)


Swin Grid Search | LR=0.001 | WD=0.001
  Epoch 1/5 | Train Acc: 0.7424 | Val Acc: 0.8713
  Epoch 2/5 | Train Acc: 0.8411 | Val Acc: 0.8580
  Epoch 3/5 | Train Acc: 0.8864 | Val Acc: 0.9000
  Epoch 4/5 | Train Acc: 0.9369 | Val Acc: 0.9393
  Epoch 5/5 | Train Acc: 0.9693 | Val Acc: 0.9400
  >> Best Val Acc: 0.9400

Swin Grid Search | LR=0.001 | WD=0.0001
  Epoch 1/5 | Train Acc: 0.7414 | Val Acc: 0.8113
  Epoch 2/5 | Train Acc: 0.8379 | Val Acc: 0.8700
  Epoch 3/5 | Train Acc: 0.8812 | Val Acc: 0.9120
  Epoch 4/5 | Train Acc: 0.9326 | Val Acc: 0.9233
  Epoch 5/5 | Train Acc: 0.9644 | Val Acc: 0.9413
  >> Best Val Acc: 0.9413

Swin Grid Search | LR=0.001 | WD=1e-05


  Epoch 1/5 | Train Acc: 0.1093 | Val Acc: 0.1440
  Epoch 2/5 | Train Acc: 0.1351 | Val Acc: 0.1553
  Epoch 3/5 | Train Acc: 0.1438 | Val Acc: 0.1820
  Epoch 4/5 | Train Acc: 0.1566 | Val Acc: 0.1533
  Epoch 5/5 | Train Acc: 0.1712 | Val Acc: 0.1833
  >> Best Val Acc: 0.1833

Swin Grid Search | LR=0.0001 | WD=0.001
  Epoch 1/5 | Train Acc: 0.8694 | Val Acc: 0.9500
  Epoch 2/5 | Train Acc: 0.9582 | Val Acc: 0.9507
  Epoch 3/5 | Train Acc: 0.9747 | Val Acc: 0.9653
  Epoch 4/5 | Train Acc: 0.9861 | Val Acc: 0.9687
  Epoch 5/5 | Train Acc: 0.9906 | Val Acc: 0.9687
  >> Best Val Acc: 0.9687

Swin Grid Search | LR=0.0001 | WD=0.0001
  Epoch 1/5 | Train Acc: 0.8840 | Val Acc: 0.9407
  Epoch 2/5 | Train Acc: 0.9575 | Val Acc: 0.9607
  Epoch 3/5 | Train Acc: 0.9755 | Val Acc: 0.9647
  Epoch 4/5 | Train Acc: 0.9839 | Val Acc: 0.9640
  Epoch 5/5 | Train Acc: 0.9915 | Val Acc: 0.9713
  >> Best Val Acc: 0.9713

Swin Grid Search | LR=0.0001 | WD=1e-05
  Epoch 1/5 | Train Acc: 0.8889 | Val Acc: 0.948

In [ ]:
# Task 2 — final training, ensemble, and submission

def train_model(model_name, lr, wd, num_epochs=FINAL_EPOCHS):
    print(f"\n{'='*60}")
    print(f"TRAINING: {model_name} | LR={lr} | WD={wd}")
    print(f"{'='*60}")

    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=NUM_CLASSES
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    val_criterion = nn.CrossEntropyLoss()

    train_accs, val_accs = [], []
    best_val_acc = 0.0

    model_tag = model_name.replace('/', '_')
    best_path = os.path.join(TASK2_SAVE_DIR, f'best_{model_tag}.pth')

    for epoch in range(num_epochs):
        model.train()
        correct = 0
        total = 0

        for images, labels in task2_train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            mixed_images, mixed_labels = mixup_cutmix(images, labels)
            outputs = model(mixed_images)

            log_probs = torch.nn.functional.log_softmax(outputs, dim=1)
            loss = -(mixed_labels * log_probs).sum(dim=1).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                clean_outputs = model(images)
                preds = clean_outputs.argmax(dim=1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        train_acc = correct / total

        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in task2_val_loader:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                outputs = model(images)
                preds = outputs.argmax(dim=1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()

        val_acc = val_correct / val_total
        current_lr = optimizer.param_groups[0]['lr']
        scheduler.step()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_path)

        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"  Epoch {epoch+1}/{num_epochs} | LR: {current_lr:.6f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    print(f"  >> Best Val Acc: {best_val_acc:.4f}")

    metrics = {
        'train_accs': train_accs,
        'val_accs': val_accs,
        'best_val_acc': best_val_acc
    }

    with open(os.path.join(TASK2_SAVE_DIR, f'metrics_{model_tag}.json'), 'w') as f:
        json.dump(metrics, f, indent=2)

    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    return model, metrics


# Use model-specific hyperparameters from grid search
model_configs = [
    ('efficientnet_b0', best_lr, best_wd),
    ('swin_small_patch4_window7_224', best_lr_swin, best_wd_swin),
]

models_task2 = []
all_metrics_task2 = {}

for name, lr, wd in model_configs:
    model, metrics = train_model(name, lr=lr, wd=wd, num_epochs=FINAL_EPOCHS)
    models_task2.append(model)
    all_metrics_task2[name] = metrics

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("ALL TASK 2 MODELS TRAINED")
print(f"{'='*60}")
for name, lr, wd in model_configs:
    print(f"  {name} (LR={lr}, WD={wd}): Best Val Acc = {all_metrics_task2[name]['best_val_acc']:.4f}")

# Ensemble validation accuracy
all_val_probs = []

for model in models_task2:
    model.eval()
    model_probs = []

    with torch.no_grad():
        for images, _ in task2_val_loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            model_probs.append(probs.cpu())

    model_probs = torch.cat(model_probs, dim=0)
    all_val_probs.append(model_probs)

ensemble_val_probs = torch.stack(all_val_probs).mean(dim=0)
ensemble_val_preds = ensemble_val_probs.argmax(dim=1)
ensemble_val_acc = (ensemble_val_preds == val_labels_task2).float().mean().item()

print(f"\n{'='*60}")
print("VALIDATION RESULTS")
print(f"{'='*60}")
print("Individual model best validation accuracies:")
for name, lr, wd in model_configs:
    print(f"  {name}: {all_metrics_task2[name]['best_val_acc']:.4f}")
print(f"\nEnsemble Validation Accuracy: {ensemble_val_acc:.4f}")

# Ensemble test predictions
all_test_probs = []

for model in models_task2:
    model.eval()
    model_probs = []

    with torch.no_grad():
        for images in task2_test_loader:
            images = images.to(DEVICE, non_blocking=True)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            model_probs.append(probs.cpu())

    model_probs = torch.cat(model_probs, dim=0)
    all_test_probs.append(model_probs)

ensemble_test_probs = torch.stack(all_test_probs).mean(dim=0)
ensemble_test_preds = ensemble_test_probs.argmax(dim=1)
pred_labels_task2 = [idx_to_class[p.item()] for p in ensemble_test_preds]

# Build submission
submission_task2 = pd.DataFrame({
    'original_index': test_csv_task2['original_index'].values,
    'label_name': pred_labels_task2
})

submission_path = os.path.join(TASK2_SAVE_DIR, 'submission_task2.csv')
submission_task2.to_csv(submission_path, index=False)

# Save summary
summary = {
    'efficientnet_b0': {'lr': best_lr, 'wd': best_wd},
    'swin_small': {'lr': best_lr_swin, 'wd': best_wd_swin},
    'individual_best_val_acc': {
        name: float(all_metrics_task2[name]['best_val_acc']) for name, _, _ in model_configs
    },
    'ensemble_val_acc': float(ensemble_val_acc)
}

with open(os.path.join(TASK2_SAVE_DIR, 'task2_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n{'='*60}")
print("SUBMISSION")
print(f"{'='*60}")
print(f"Saved to: {submission_path}")
print(f"Total predictions: {len(submission_task2)}")
print(submission_task2.head(10))



TRAINING: efficientnet_b0 | LR=0.001 | WD=0.0001
  Epoch 1/20 | LR: 0.001000 | Train Acc: 0.8385 | Val Acc: 0.8693
  Epoch 2/20 | LR: 0.000994 | Train Acc: 0.9167 | Val Acc: 0.8967
  Epoch 3/20 | LR: 0.000976 | Train Acc: 0.9433 | Val Acc: 0.9167
  Epoch 4/20 | LR: 0.000946 | Train Acc: 0.9593 | Val Acc: 0.9053
  Epoch 5/20 | LR: 0.000905 | Train Acc: 0.9676 | Val Acc: 0.9193
  Epoch 6/20 | LR: 0.000854 | Train Acc: 0.9765 | Val Acc: 0.9240
  Epoch 7/20 | LR: 0.000794 | Train Acc: 0.9814 | Val Acc: 0.9240
  Epoch 8/20 | LR: 0.000727 | Train Acc: 0.9887 | Val Acc: 0.9313
  Epoch 9/20 | LR: 0.000655 | Train Acc: 0.9887 | Val Acc: 0.9220
  Epoch 10/20 | LR: 0.000578 | Train Acc: 0.9912 | Val Acc: 0.9307
  Epoch 11/20 | LR: 0.000500 | Train Acc: 0.9925 | Val Acc: 0.9340
  Epoch 12/20 | LR: 0.000422 | Train Acc: 0.9952 | Val Acc: 0.9400
  Epoch 13/20 | LR: 0.000345 | Train Acc: 0.9949 | Val Acc: 0.9380
  Epoch 14/20 | LR: 0.000273 | Train Acc: 0.9966 | Val Acc: 0.9453
  Epoch 15/20 | LR: 0